# 01 - MCP

In this notebook we'll be exploring the Model Context Protocol (MCP). 
* First we'll cover some background on MCP - what is MCP and why it might be useful to us.
* Then we'll build our very own MCP server and connect it to our agent
* Finally, we'll learn how to connect our agent to the wide library of publicly available MCP servers


## What is Model Context Protocol (MCP)?

There is a lot of hype around the Model Context Protocol (MCP) after Anthropic introduced it in Oct 2024 as **an open-standard, "plug-and-play" framework designed to simplify how AI models connect to data sources and local tools**, replacing fragmented integrations with a universal standard for seamless data access and tool use.

For a more detailed coverage see [MCP explained](mcp_explained.md)

You can find a lot of public MCP servers at the following links:

* https://github.com/modelcontextprotocol/servers
* https://github.com/punkpeye/awesome-mcp-servers 
* https://modelcontextprotocol.io/servers/
* https://smithery.ai/
* https://mcp.so/servers 


Think of MCP as a USB connector that we are very familiar with. It allows you to iterface, say your laptop, with several other perepherals without really worrying about how the connect works - it just works! Without the USB standard, each perepheral would come with it's own adapter, cable etc. A nightmare! In fact, Apple was rather infamously doing just this until the EU forced it to standarize their iPhone chargers (which now default to USB 3 protocols!), which means you can use ant standard charger with your latest iPhone.

Consider another example of how the JDBC/ODBC protocols decoupled applications from databases. Before JDBC/ODBC, if you wanted to switch your application from an Oracle database to an MS SQL database, you often had to rewrite the entire data-access layer of your code because the libraries, function calls, and error handling were completely different. With JDBC/ODBC, you may have to just change the way you connect to the database - rest of the interface API remains the same. Similarly with agents before the advent of MCP - if you wanted to give an AI agent access to "Google Drive," you had to write custom integration code specifically for that agent. If you then moved the "drive" to Microsoft 365, you had to write it all over again. MCP turns "tools" into plug-and-play servers.

Also, consider a custom tool (function) you built for your Agent. Let's say it handles a payment gateway interface enabling your Agent to send & receive payments. Say another developer in your organization (or another organization) wants to enable similar functionality, the she would have to write a similar tool. What if you wrote a MCP server as a standard Payment Gateway? Now suddenly all developers in your organization use the same server to enable their agents with Payments functionality - no code duplication. If your MCP server then goes "public" all developers everywhere can use the same functionality without writing custom tools!

In all the examples above, your application is a _host_ that uses a _client_ (interfacing code - JDBC/ODBC) to interface with the _server_ (Database). 


Now let's drill down to some specifics:
* An MCP host _hosts_ an MCP client, which commun

## Using Local MCP Server

In [15]:
import sys
import asyncio
from dotenv import load_dotenv
from rich.console import Console

load_dotenv(override=True)
console = Console()

> **NOTE**:
>
> There is a **known Windows + Jupyter notebook incompatibility issue** when MCP tries to launch subprocess-based servers, like we will do in this notebook.<br/><br/>


If you **do not** implement the fix in the next cell and run the cells thereafter you'll encounter two cascading errors:
1. **Error 1:** `NotImplementedError` (async subprocess)
MCP first tries `anyio.open_process()` → `asyncio.create_subprocess_exec()`.

On Windows, async subprocess creation requires `ProactorEventLoop`. Jupyter notebooks run their own event loop (typically a `SelectorEventLoop` or one managed by `nest_asyncio`), which doesn't support subprocesses. Therefore base `class BaseEventLoop._make_subprocess_transport` explicitly raises `NotImplementedError` for this case.

2. **Error 2:** `UnsupportedOperation: fileno` (sync fallback)
MCP catches the first error and falls back to plain `subprocess.Popen`. This fails because:

* Jupyter redirects and wraps `stdout/stderr` with custom stream objects (to capture cell output)
* These wrapper streams don't have a real OS file descriptor, so `fileno()` raises `UnsupportedOperation`
`subprocess.Popen` needs a real file descriptor when you pass `stderr=errlog`

To fix these errors, we must run the code in the following cell **_before_** calling any MCP server code. It has a `sys.platform == "win32"` _guard_, which means it will run only if your OS is Windows.

In [16]:
# Fix for Windows issues in Jupyter notebooks
if sys.platform == "win32":
    # 1. Use ProactorEventLoop for subprocess support
    if not isinstance(
        asyncio.get_event_loop_policy(), asyncio.WindowsProactorEventLoopPolicy
    ):
        asyncio.set_event_loop_policy(asyncio.WindowsProactorEventLoopPolicy())

    # 2. Redirect stderr to avoid fileno() error when launching MCP servers
    if "ipykernel" in sys.modules:
        sys.stderr = sys.__stderr__

In [17]:
from langchain_mcp_adapters.client import MultiServerMCPClient

client = MultiServerMCPClient(
    {
        "local_server": {
            "transport": "stdio",
            "command": "python",
            "args": ["resources/01_mcp_server.py"],
        },
    }
)

In [18]:
# get the tools, resources & prompts that our local server provides
tools = await client.get_tools()
print("-------- list of tools available -------- ")
console.print(tools)

# get resources
resources = await client.get_resources("local_server")
print("-------- resources available -------- ")
console.print(resources)

# get prompts
prompt = await client.get_prompt("local_server", "prompt")
prompt = prompt[0].content
print("-------- prompt available -------- ")
console.print(prompt)

-------- list of tools available -------- 


[
    StructuredTool(
        name='web_search',
        description='Search the web for information based on query',
        args_schema={
            'properties': {'query': {'title': 'Query', 'type': 'string'}},
            'required': ['query'],
            'title': 'web_searchArguments',
            'type': 'object'
        },
        response_format='content_and_artifact',
        coroutine=<function convert_mcp_tool_to_langchain_tool.<locals>.call_tool at 0x000001E18C15D260>
    )
]

-------- resources available -------- 


[
    Blob(
        metadata={'uri': AnyUrl('github://langchain-ai/langchain-mcp-adapters/blob/main/README.md')},
        data='# LangChain MCP Adapters\n\nThis library provides a lightweight wrapper that makes [Anthropic Model 
Context Protocol (MCP)](https://modelcontextprotocol.io/introduction) tools compatible with 
[LangChain](https://github.com/langchain-ai/langchain) and 
[LangGraph](https://github.com/langchain-ai/langgraph).\n\n![MCP](static/img/mcp.png)\n\n> [!note]\n> A 
JavaScript/TypeScript version of this library is also available at 
[langchainjs](https://github.com/langchain-ai/langchainjs/tree/main/libs/langchain-mcp-adapters/).\n\n## 
Features\n\n- 🛠️ Convert MCP tools into [LangChain tools](https://python.langchain.com/docs/concepts/tools/) that 
can be used with [LangGraph](https://github.com/langchain-ai/langgraph) agents\n- 📦 A client implementation that 
allows you to connect to multiple MCP servers and load tools from them\n\n## Installation\n\n```bash\npip install 
langchain-mcp-adapters\n```\n\n## Quickstart\n\nHere is a simple example of using the MCP tools with a LangGraph 
agent.\n\n```bash\npip install langchain-mcp-adapters langgraph "langchain[openai]"\n\nexport 
OPENAI_API_KEY=<your_api_key>\n```\n\n### Server\n\nFirst, let\'s create an MCP server that can add and multiply 
numbers.\n\n```python\n# math_server.py\nfrom mcp.server.fastmcp import FastMCP\n\nmcp = 
FastMCP("Math")\n\n@mcp.tool()\ndef add(a: int, b: int) -> int:\n    """Add two numbers"""\n    return a + 
b\n\n@mcp.tool()\ndef multiply(a: int, b: int) -> int:\n    """Multiply two numbers"""\n    return a * b\n\nif 
__name__ == "__main__":\n    mcp.run(transport="stdio")\n```\n\n### Client\n\n```python\n# Create server parameters
for stdio connection\nfrom mcp import ClientSession, StdioServerParameters\nfrom mcp.client.stdio import 
stdio_client\n\nfrom langchain_mcp_adapters.tools import load_mcp_tools\nfrom langchain.agents import 
create_agent\n\nserver_params = StdioServerParameters(\n    command="python",\n    # Make sure to update to the 
full absolute path to your math_server.py file\n    args=["/path/to/math_server.py"],\n)\n\nasync with 
stdio_client(server_params) as (read, write):\n    async with ClientSession(read, write) as session:\n        # 
Initialize the connection\n        await session.initialize()\n\n        # Get tools\n        tools = await 
load_mcp_tools(session)\n\n        # Create and run the agent\n        agent = create_agent("openai:gpt-4.1", 
tools)\n        agent_response = await agent.ainvoke({"messages": "what\'s (3 + 5) x 12?"})\n```\n\n## Multiple MCP
Servers\n\nThe library also allows you to connect to multiple MCP servers and load tools from them:\n\n### 
Server\n\n```python\n# math_server.py\n...\n\n# weather_server.py\nfrom typing import List\nfrom mcp.server.fastmcp
import FastMCP\n\nmcp = FastMCP("Weather")\n\n@mcp.tool()\nasync def get_weather(location: str) -> str:\n    """Get
weather for location."""\n    return "It\'s always sunny in New York"\n\nif __name__ == "__main__":\n    
mcp.run(transport="http")\n```\n\n```bash\npython weather_server.py\n```\n\n### Client\n\n```python\nfrom 
langchain_mcp_adapters.client import MultiServerMCPClient\nfrom langchain.agents import create_agent\n\nclient = 
MultiServerMCPClient(\n    {\n        "math": {\n            "command": "python",\n            # Make sure to 
update to the full absolute path to your math_server.py file\n            "args": ["/path/to/math_server.py"],\n   
"transport": "stdio",\n        },\n        "weather": {\n            # Make sure you start your weather server on 
port 8000\n            "url": "http://localhost:8000/mcp",\n            "transport": "http",\n        }\n    
}\n)\ntools = await client.get_tools()\nagent = create_agent("openai:gpt-4.1", tools)\nmath_response = await 
agent.ainvoke({"messages": "what\'s (3 + 5) x 12?"})\nweather_response = await agent.ainvoke({"messages": "what is 
the weather in nyc?"})\n``

-------- prompt available -------- 


You are a helpful assistant that answers questions about LangChain, LangGraph and LangSmith.
    
    You can use the following tools/resources to answe user's questions:
    - search_web: search the web for information.
    - github_file: Access the langchain-ai repo files.

    If user user asks a question that is NOT RELATED to LangChain, LangGraph or LangSmith, you
    should respond with "I am sorry, I can only answer questions related to LangChain, LangGraph and LangSmith. 
Please ask a relevant question."

    You may try multiple tools and resource calls to answer the user's question.

    You may also ask clarifying questions to the user to better understand their question before responding.

In [19]:
# create our agent the usual way
from langchain.agents import create_agent

agent = create_agent(
    model="openai:gpt-5-nano",
    # tools from MCP server - is a list!
    tools=tools,
    # system prompt also from MCP server
    system_prompt=prompt,
)

Now let's ask it some questions about LangChain/LangGraph/LangSmith

In [20]:
from langchain.messages import HumanMessage

config = {"configurable": {"thread_id": "1"}}

# NOTE: we use ainvoke() instead of invoke()
response = await agent.ainvoke(
    {
        "messages": [
            HumanMessage(content="Tell me about the langchain-mcp-adapters library")
        ]
    },
    config=config,
)
print(" ----- Response from MCP server ----- ")
# to view all the messages exchanges use the following 2 lines
# from pprint import pprint
# pprint(response)
print(response["messages"][-1].content)

 ----- Response from MCP server ----- 
Here’s a concise overview of the langchain-mcp-adapters library and what it’s for.

What it is
- A library that bridges Anthropic’s Model Context Protocol (MCP) with LangChain and LangGraph.
- It provides adapters to convert MCP tools, prompts, and resources into LangChain-LangGraph compatible forms, so MCP tool servers can be used inside LangChain agents and workflows.
- Designed to let agents pull from many MCP servers at once, without writing custom adapters for each server.

Why you’d use it
- To easily leverage a large ecosystem of MCP tool servers within LangChain/LangGraph.
- To avoid hand-rolling MCP adapters for every tool server; the library handles conversion and integration.
- To run multi-server MCP tool sets in a single agent or graph, enabling more powerful tool combinations.

Key features
- Tools adapter: converts MCP tools into LangChain-compatible tools.
- Prompts adapter: translates MCP prompts into LangChain messages.
- Resourc

Cool! It was able to get me a response from the local MCP server.

## Online MCP
Next we'll attempt to use an online MCP time server that tells you the time in New York.

In [21]:
client = MultiServerMCPClient(
    {
        "time": {
            "transport": "stdio",
            "command": "uvx",
            "args": ["mcp-server-time", "--local-timezone=America/New_York"],
        }
    }
)

online_tools = await client.get_tools()
print("-------- list of tools available -------- ")
console.print(online_tools)

-------- list of tools available -------- 


[
    StructuredTool(
        name='get_current_time',
        description='Get current time in a specific timezones',
        args_schema={
            'type': 'object',
            'properties': {
                'timezone': {
                    'type': 'string',
                    'description': "IANA timezone name (e.g., 'America/New_York', 'Europe/London'). Use 
'America/New_York' as local timezone if no timezone provided by the user."
                }
            },
            'required': ['timezone']
        },
        response_format='content_and_artifact',
        coroutine=<function convert_mcp_tool_to_langchain_tool.<locals>.call_tool at 0x000001E18C15F6A0>
    ),
    StructuredTool(
        name='convert_time',
        description='Convert time between timezones',
        args_schema={
            'type': 'object',
            'properties': {
                'source_timezone': {
                    'type': 'string',
                    'description': "Source IANA timezone name (e.g., 'America/New_York', 'Europe/London'). Use 
'America/New_York' as local timezone if no source timezone provided by the user."
                },
                'time': {'type': 'string', 'description': 'Time to convert in 24-hour format (HH:MM)'},
                'target_timezone': {
                    'type': 'string',
                    'description': "Target IANA timezone name (e.g., 'Asia/Tokyo', 'America/San_Francisco'). Use 
'America/New_York' as local timezone if no target timezone provided by the user."
                }
            },
            'required': ['source_timezone', 'time', 'target_timezone']
        },
        response_format='content_and_artifact',
        coroutine=<function convert_mcp_tool_to_langchain_tool.<locals>.call_tool at 0x000001E18C23EA20>
    )
]

In [22]:
time_agent = create_agent(
    model="openai:gpt-5-nano",
    tools=online_tools,
    system_prompt="""You can fetch time from New York using tools available 
    to you. Display just the time & nothing else. Use the following format:

    It’s 12:36 PM on Friday, June 28, 2019 in New York (Eastern Daylight Time, UTC-4)
    """,
)

In [23]:
# let's ask for time in New York
question = HumanMessage(content="What time is it?")

response = await time_agent.ainvoke({"messages": [question]})

print(" ----- Response from MCP Time server ----- ")
# to view all the messages exchanges use the following 2 lines
# from pprint import pprint
# pprint(response)
print(response["messages"][-1].content)

 ----- Response from MCP Time server ----- 
It’s 10:30 AM on Friday, April 17, 2026 in New York (Eastern Daylight Time, UTC-4)


In [24]:
# an yet another free flights MCP server
kiwi_client = MultiServerMCPClient(
    {
        "travel_server": {
            "transport": "streamable_http",
            "url": "https://mcp.kiwi.com",
        }
    }
)

kiwi_tools = await kiwi_client.get_tools()
print("-------- list of tools available -------- ")
console.print(kiwi_tools)

-------- list of tools available -------- 


[
    StructuredTool(
        name='search-flight',
        description='\n# Search for a flight\n\n## Description\n\nUses the Kiwi API to search for available flights
between two locations on a specific date.\n\n## How it works\n\nThe tool will:\n1. Search for matching locations to
resolve airport codes\n2. Find available flights for the specified route and date range\n\n## Method\n\nCall this 
tool whenever a user wants to search for flights, regardless of whether they provided exact airport codes or just 
city names.\n\nYou should display the returned results in a markdown table format: Group the results by price 
(those who are the cheapest), duration (those who are the shortest, i.e. have the smallest 
\'totalDurationInSeconds\') and the rest (those that could still be interesting).\n\nAlways display for each flight
in order:\n  - In the 1st column: The departure and arrival airports, including layovers (e.g. "Paris CDG → 
Barcelona BCN → Lisbon LIS")\n  - In the 2nd column: The departure and arrival dates & times in the local 
timezones, and duration of the flight (e.g. "03/08 06:05 → 09:30 (3h 25m)", use \'durationInSeconds\' to display 
the duration and not \'totalDurationInSeconds\')\n  - In the 3rd column: The cabin class (e.g. "Economy")\n  - (In 
case of return flight only) In the 4th column: The return flight departure and arrival airports, including layovers
(e.g. "Paris CDG → Barcelona BCN → Lisbon LIS")\n  - (In case of return flight only) In the 5th column: The return 
flight departure and arrival dates & times in the local timezones, and duration of the flight (e.g. "03/08 06:05 → 
09:30 (3h 25m)", use \'return.durationInSeconds\' to display the duration)\n  - (In case of return flight only) In 
the 6th column: The return flight cabin class (e.g. "Economy")\n  - In the previous-to-last column: The total price
of the flight\n  - In the last column: The deep link to book the flight\n\nFinally, provide a summary highlighting 
the best prices, the shortest flights and a recommendation. End wishing a nice trip to the user with a short fun 
fact about the destination!\n',
        args_schema={
            'type': 'object',
            'properties': {
                'flyFrom': {
                    'type': 'string',
                    'minLength': 1,
                    'description': 'Location to fly from: It could be a city or an airport name or code'
                },
                'flyTo': {
                    'type': 'string',
                    'description': 'Location to fly to: It could be a city or an airport name or code'
                },
                'departureDate': {
                    'type': 'string',
                    'pattern': '^\\d{2}\\/\\d{2}\\/\\d{4}$',
                    'description': 'Departure date in dd/mm/yyyy format'
                },
                'departureDateFlexRange': {
                    'type': 'integer',
                    'minimum': 0,
                    'maximum': 3,
                    'default': 0,
                    'description': 'Departure date flexibility range in days (0 to 3 days before/after the selected
departure date)'
                },
                'returnDate': {
                    'type': 'string',
                    'pattern': '^\\d{2}\\/\\d{2}\\/\\d{4}$',
                    'description': 'Return date in dd/mm/yyyy format'
                },
                'returnDateFlexRange': {
                    'type': 'integer',
                    'minimum': 0,
                    'maximum': 3,
                    'default': 0,
                    'description': 'Return date flexibility range in days (0 to 3 days before/after the selected 
return date)'
                },
                'passengers': {
                    'type': 'object',
                    'properties': {
                        'adults': {
                            'type': 'integer',
                            'minimum': 0,
                            'maximum': 9,


In [28]:
# and call it
from langgraph.checkpoint.memory import InMemorySaver

travel_agent = create_agent(
    model="openai:gpt-5-nano",
    tools=kiwi_tools,
    checkpointer=InMemorySaver(),
    system_prompt="You are a travel agent. No follow up questions!",
)

In [ ]:
# let's ask for time in New York
question = HumanMessage(
    content="Get me a one-way direct flight from San Francisco to Tokyo on 30/04/2026"
)

config = {"configurable": {"thread_id": "2"}}

response = await travel_agent.ainvoke({"messages": [question]}, config=config)

print(" ----- Response from Kiwi server ----- ")
# to view all the messages exchanges use the following 2 lines
# from pprint import pprint
# pprint(response)
print(response["messages"][-1].content)